**Ways to build NGrams**

**Table of contents**<a id='toc0_'></a>    
- [**Mock Data & Config**](#toc1_1_)    
- [**NGram - Torch**](#toc2_)    
- [**NGRAM Sklearn**](#toc3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[**Mock Data & Config**](#toc0_)

In [2]:
true_sentence_labels = [
    # 0
    "bring it closer my family is not hungry",
    # 1
    "please bring my computer here",
    # 2
    "i hope this version works better than the last one",
    # 3
    "today the weather is very nice outside",
    # 4
    "can you help me find my glasses",
    # 5
    "i am not feeling very well today",
    # 6
    "they are going to visit us tomorrow",
    # 7
    "my sister likes to listen to music",
    # 8
    "we need to leave the house soon",
    # 9
    "this experiment is harder than i expected"
]

pred_sentence_labels = [
    # 0
    "BRIHNG IHT KLOWSER MAY FAEMAHLIY IHZ NAAT HHAHNGGRIY",
    # 1
    "PLIYZ BRIHNG MAY KAHMPYUWTER HHIYR",
    # 2
    "AY HHOWP DHIS VERZHAN WERKS BEHTER DHAEN DHAE LAEST WAHN",
    # 3
    "TUWDEY DHAH WEHDHER IHZ VEHRIY NAYS AWTSAYD",
    # 4
    "KAEN YUW HHEHLP MIY FAYND MAY GLAESAHZ",
    # 5
    "AY AEM NAAT FIYLING VEHRIY WEHL TUHDEY",
    # 6
    "DHEY AAR GOWIHNG TUW VIHZIHT AHS TUHMARO",
    # 7
    "MAY SIHSSTER LAYKS TUW LIHSEN TUW MYUWZIHK",
    # 8
    "WIY NIYD TUW LIYV DHAH HAWS SUWN",
    # 9
    "DHIS EHKSPERIHMEN IHZ HHAARDR DHAEN AY IHKSPHYEKTIHD"
]


# <a id='toc2_'></a>[**NGram - Torch**](#toc0_)

In [ ]:
# NGRAM

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.notebook import tqdm  # or: from tqdm import tqdm

CONTEXT_SIZE = 5
EMBEDDING_DIM = 128 # int(2 ** CONTEXT_SIZE)
EPOCH = 5
L_RATE = 0.001
PAD_TOKEN = "<pad>"  # or "<s>"

class NGramProcessor:
    """NGram Data Processor"""

    def __init__(self, observed, context_size: int = CONTEXT_SIZE, pad_token: str = PAD_TOKEN):
        self.context_size = context_size
        self.pad_token = pad_token

        # pad on the left so we can form contexts for the first tokens
        self.tokens = self._pad_sequence(observed)
        self.ngrams = self.preprocess(self.tokens)

        self.vocab = set(self.tokens)
        self.tag2id = {word: i for i, word in enumerate(self.vocab)}
        self.id2tag = {i: word for i, word in enumerate(self.vocab)}

    def __len__(self):
        return len(self.vocab)

    def _pad_sequence(self, observed):
        """Prepend pad_token context_size times."""
        return [self.pad_token] * self.context_size + list(observed)

    def preprocess(self, tokens):
        """
        Build n-grams: ([w_{i-context_size}, ..., w_{i-1}], w_i)
        with i starting at context_size.
        """
        ngrams = []
        for i in range(self.context_size, len(tokens)):
            context = [tokens[i - j - 1] for j in range(self.context_size)]
            target = tokens[i]
            ngrams.append((context, target))
        return ngrams

class NGramLanguageModeler(nn.Module):
    """ NGramLanguageModeler creates an key-value querying matrix via Pytorch's Embedding layer. """

    def __init__(self, vocab_size, embedding_dim, context_size):
        super(NGramLanguageModeler, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, 128)
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view((1, -1))
        out = F.relu(self.linear1(embeds))
        out = self.linear2(out)
        log_probs = F.log_softmax(out, dim=1)
        return log_probs

def trainer(train_text, epoch: int = EPOCH, lr: float = L_RATE, context_size: int = CONTEXT_SIZE, embedding_dim: int = EMBEDDING_DIM):

    vocab = NGramProcessor(train_text)
    loss_function = nn.NLLLoss()
    model = NGramLanguageModeler(len(vocab), embedding_dim, context_size)
    optimizer = optim.SGD(model.parameters(), lr=lr)

    losses = []

    epoch_bar = tqdm(range(epoch), desc="Training phoneme Ngram Readable Sentences", leave=True)

    for ep in epoch_bar:
        total_loss = 0
        for context, target in vocab.ngrams:
            # Step 1. Prepare the inputs to be passed to the model (i.e, turn the words
            # into integer indices and wrap them in tensors)
            context_idxs = torch.tensor([vocab.tag2id[w] for w in context], dtype=torch.long)

            # Step 2. Recall that torch *accumulates* gradients. Before passing in a
            # new instance, you need to zero out the gradients from the old
            # instance
            model.zero_grad()

            # Step 3. Run the forward pass, getting log probabilities over next
            # words
            log_probs = model(context_idxs)

            # Step 4. Compute your loss function. (Again, Torch wants the target
            # word wrapped in a tensor)
            loss = loss_function(log_probs, torch.tensor([vocab.tag2id[target]], dtype=torch.long))

            # Step 5. Do the backward pass and update the gradient
            loss.backward()
            optimizer.step()

            # Get the Python number from a 1-element Tensor by calling tensor.item()
            total_loss += loss.item()

        losses.append(total_loss)
        epoch_bar.set_description(f"Epoch {ep+1}/{epoch} - loss: {total_loss:.4f}")
        epoch_bar.refresh()

    # print(losses)
    # To get the embedding of a particular word, e.g. "beauty"
    # print(model.embeddings.weight[vocab.tag2id["beauty"]])
    return vocab, model

def evaluate(model, vocab, eval_text):
    """
    eval_text: list of tokens (same format you used to build vocab/ngrams)
    vocab: KnowledgeBase object
    """
    model.eval()
    loss_fn = nn.NLLLoss(reduction="sum")  # sum for average later
    total_loss = 0
    total_tokens = 0

    # tqdm over evaluation ngrams
    for context, target in tqdm(eval_text, desc="Evaluating", leave=True):
        context_idxs = torch.tensor(
            [vocab.tag2id[w] for w in context],
            dtype=torch.long
        )

        with torch.no_grad():
            log_probs = model(context_idxs)
            loss = loss_fn(
                log_probs,
                torch.tensor([vocab.tag2id[target]], dtype=torch.long)
            )

        total_tokens += 1
        total_loss += loss.item()

    if total_tokens == 0:
        return {"avg_loss": float("nan"), "perplexity": float("nan")}

    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss)

    return {"avg_loss": avg_loss, "perplexity": ppl}

def predict_next(model, vocab: NGramProcessor, context_tokens):
    model.eval()
    ctx = list(context_tokens)
    if len(ctx) < vocab.context_size:
        ctx = [vocab.pad_token] * (vocab.context_size - len(ctx)) + ctx
    else:
        ctx = ctx[-vocab.context_size:]

    ctx_ids = torch.tensor([vocab.tag2id[w] for w in ctx], dtype=torch.long)
    with torch.no_grad():
        log_probs = model(ctx_ids)
        next_id = log_probs.argmax(dim=1).item()
    return vocab.id2tag[next_id]

true_sample = [w for s in true_sentence_labels for w in s.split()]
vocab, ngram_model = trainer(train_text=true_sample,
                             context_size=CONTEXT_SIZE,
                             embedding_dim=EMBEDDING_DIM)

Training phoneme Ngram Readable Sentences:   0%|          | 0/5 [00:00<?, ?it/s]

In [68]:
# one-step prediction
prefix = ["bring"]
next_word = predict_next(ngram_model, vocab, prefix)
print("Next word:", next_word)

Next word: bring


# <a id='toc3_'></a>[**NGRAM Sklearn**](#toc0_)

In [ ]:
import re
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier

BOS = "<s>"
UNK = "<unk>"
EOS = "</s>"
CONTEXT_SIZE = 2
TOP_K = 5
N_FEATURES = 2**20
BATCH_SIZE = 256

def tokenize(text: str):
    return re.findall(r"[A-Za-z']+|[0-9]+|[^\sA-Za-z0-9]", text.lower())


def make_training_pairs(texts, n_context=CONTEXT_SIZE, vocab=None):
    """
    Yields (context_string, next_word) pairs.
    Optionally maps OOV words to UNK if vocab is provided.
    """
    for t in texts:
        toks = tokenize(t)
        if vocab is not None:
            toks = [w if w in vocab else UNK for w in toks]
        toks = [BOS] * n_context + toks
        for i in range(n_context, len(toks)):
            ctx = " ".join(toks[i - n_context:i])
            nxt = toks[i]
            yield ctx, nxt


class OnlineNGramClassifier:
    def __init__(self, base_texts, n_context=CONTEXT_SIZE, n_features=N_FEATURES):
        """
        base_texts: initial corpus to define vocab/classes (true sentences).
        """
        self.n_context = n_context
        self.n_features = n_features

        # fixed vocab and classes for partial_fit
        vocab = {BOS, UNK}
        for t in base_texts:
            vocab.update(tokenize(t))
        self.vocab = vocab
        self.classes = np.array(sorted(self.vocab))

        self.vec = HashingVectorizer(
            n_features=self.n_features,
            alternate_sign=False,
            norm="l2",
        )
        self.clf = SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            learning_rate="optimal",
        )
        self._is_fitted = False

    def partial_fit(self, texts, batch_size=BATCH_SIZE):
        """
        Incrementally fit on a list (or iterable) of new sentences.
        """
        X_batch, y_batch = [], []

        for ctx, nxt in make_training_pairs(texts, n_context=self.n_context, vocab=self.vocab):
            X_batch.append(ctx)
            y_batch.append(nxt)

            if len(X_batch) >= batch_size:
                X = self.vec.transform(X_batch)
                y = np.array(y_batch)
                if not self._is_fitted:
                    self.clf.partial_fit(X, y, classes=self.classes)
                    self._is_fitted = True
                else:
                    self.clf.partial_fit(X, y)
                X_batch, y_batch = [], []

        # flush remainder
        if X_batch:
            X = self.vec.transform(X_batch)
            y = np.array(y_batch)
            if not self._is_fitted:
                self.clf.partial_fit(X, y, classes=self.classes)
                self._is_fitted = True
            else:
                self.clf.partial_fit(X, y)

    def predict_next(self, last_words, topk=5):
        """
        last_words: list of previous tokens (e.g. ["bring", "it"])
        returns list of (token, prob).
        """
        if not self._is_fitted:
            raise RuntimeError("Model not fitted yet. Call partial_fit first.")

        toks = [w if w in self.vocab else UNK for w in last_words][-self.n_context:]
        ctx = " ".join(([BOS] * (self.n_context - len(toks))) + toks)
        X = self.vec.transform([ctx])
        proba = self.clf.predict_proba(X)[0]
        idx = np.argsort(proba)[::-1][:topk]
        return [(self.classes[i], float(proba[i])) for i in idx]

    # ---------- persistence ----------
    def save(self, path: str | Path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        state = {
            "n_context": self.n_context,
            "n_features": self.n_features,
            "vocab": list(self.vocab),
            "classes": self.classes,
            "clf": self.clf,
            # HashingVectorizer has no learned state, but keep config if needed
        }
        with open(path, "wb") as f:
            pickle.dump(state, f, protocol=pickle.HIGHEST_PROTOCOL)

    @classmethod
    def load(cls, path: str | Path) -> "OnlineNGramClassifier":
        path = Path(path)
        with open(path, "rb") as f:
            state = pickle.load(f)

        # reconstruct with dummy base_texts; we will overwrite attributes
        obj = cls(base_texts=[], n_context=state["n_context"], n_features=state["n_features"])
        obj.vocab = set(state["vocab"])
        obj.classes = state["classes"]
        obj.clf = state["clf"]
        obj._is_fitted = True  # loaded model is already trained
        # recreate vectorizer with same settings
        obj.vec = HashingVectorizer(
            n_features=state["n_features"],
            alternate_sign=False,
            norm="l2",
        )
        return obj

# initial fit on your existing corpus
base_texts = true_sentence_labels   # list of clean sentences
online_lm = OnlineNGramClassifier(base_texts, n_context=CONTEXT_SIZE)
online_lm.partial_fit(base_texts)

print(online_lm.predict_next(["bring", "it"]))

# later, when new decoded/clean pairs arrive:
new_texts_batch_1 = ["this is a new sentence", "another new example"]
online_lm.partial_fit(new_texts_batch_1)

print(online_lm.predict_next(["bring", "it"]))

[('closer', 0.9339147668258231), ('the', 0.013742245323262766), ('listen', 0.008824523988554945), ('version', 0.004151991755048275), ('us', 0.004044792766717092)]
[('closer', 0.9373009498774025), ('the', 0.012410797309733924), ('listen', 0.008194080289097204), ('version', 0.0038482216036588397), ('us', 0.003775566002370757)]
